die aktuelle Methode hat ein strukturelles Problem: reines in-Substring-Matching ohne Wortgrenzen. Konkrete Verbesserungen, geordnet nach Aufwand/Nutzen:
1. Wortgrenzen statt Substring-Matching (wichtigster Fix)
Statt keyword in text → re.search(rf'\b{keyword}\b', text). Das behebt light/lightning, ram/frame, pen/opening, ios/studios mit einem Schlag, ohne jedes Keyword einzeln reparieren zu müssen.
2. Reihenfolge nach Spezifität, nicht nach Themenblock
Aktuell steht "Smart Home" vor "Accessories" – das ist Zufall, keine bewusste Priorisierung. Regel: mehrdeutige/kurze Keywords (tv, pen, light) sollten zuletzt geprüft werden, eindeutige Mehrwort-Begriffe (philips hue, graphics tablet) zuerst.
3. type-Spalte als zweite, unabhängige Quelle nutzen
Ihr habt in der Übung schon gesehen: type-Codes bilden teils schon saubere Kategorien (Handyhüllen, Server). Kreuzvergleich: Wenn Keyword-Kategorie und type-basierte Kategorie widersprüchlich sind → Kandidat für manuelle Prüfung. Deutlich robuster als eine einzelne Quelle blind zu vertrauen.
4. Kein 'Other'-Anteil von 0 % als Ziel
0 % ist kein Qualitätsmerkmal, sondern ein Warnsignal (siehe eben). Ziel sollte ein kleiner, aber echter Rest sein, den ihr bewusst manuell nachprüft – nicht künstlich auf 0 % gedrückt durch zu generische Keywords.
5. Validierungs-Loop statt einmalig schreiben
Nach jeder Regel-Änderung: .sample(15) pro Kategorie ansehen (macht ihr ja gerade), bevor die nächste Berechnung darauf aufbaut.
Soll ich die bestehende category_rules-Liste mit Wortgrenzen-Regex (Punkt 1) und angepasster Reihenfolge (Punkt 2) für euch umschreiben, oder wollt ihr das erst an den beiden Verifizierungs-Zellen von eben selbst nachvollziehen?

In [ ]:
# ============================================================
# Kategorien primär über 'type' (eindeutiger als reines Keyword-Matching)
# ============================================================

# 1. Herausfinden, welche type-Codes den Großteil der Produkte ausmachen
type_counts = products_qu['type'].value_counts()
n = 30
coverage = (type_counts.head(n).sum() / products_qu.shape[0] * 100).round(2)
print(f"Die {n} häufigsten type-Codes decken {coverage}% aller Produkte ab.")
type_counts.head(n)
python

# 2. Für jeden großen type-Code eine Stichprobe ansehen, um die Kategorie zu bestimmen
# (manuell durchgehen, einzeln oder in einer Schleife)
for t in type_counts.head(n).index:
    print(f"\n--- type = {t} ({type_counts[t]} Produkte) ---")
    display(products_qu.loc[products_qu['type'] == t, 'desc'].sample(min(5, type_counts[t])))
python

# 3. Mapping type -> category von Hand eintragen (auf Basis der Stichproben aus Schritt 2)
# WICHTIG: Diese Zuordnung müsst ihr selbst anhand eurer echten Stichproben ausfüllen —
# ich kann sie nicht erraten, ohne eure Daten gesehen zu haben.
type_to_category = {
    # "11865403": "Accessories & Cables",   # Beispiel: Handyhüllen
    # "12175397": "Storage & NAS",          # Beispiel: Server
    # ... weitere type-Codes ergänzen
}

products_qu['category'] = products_qu['type'].map(type_to_category)
python# 4. Für alle type-Codes OHNE manuelles Mapping (die "anderen 20%"):

#    Fallback über Keyword-Matching MIT Wortgrenzen (behebt light/lightning, ram/frame etc.)
import re

category_rules = [
    ('Computers (MacBook)', ['macbook']),
    ('Computers (iMac)', ['imac']),
    ('Computers (Mac Mini)', ['mac mini']),
    ('Mobile Phones', ['iphone']),
    ('Tablets', ['ipad', 'wacom', 'intuos', 'graphics tablet']),
    ('Wearables', ['airpods', 'watch', 'fitbit', 'bracelet']),
    ('Audio', ['speaker', 'headphone', 'microphone', 'soundbar']),
    ('Monitors', ['monitor', 'display', 'cinema display']),
    ('Storage & NAS', ['nas', 'synology', 'seagate', 'raid', 'qnap']),
    ('Components (RAM/SSD)', ['ram', 'ssd', 'hard disk', 'memory card', 'sata']),
    ('Network', ['router', 'ethernet', 'wifi', 'wi-fi', 'bluetooth']),
    ('Smart Home', ['philips hue', 'thermostat', 'homekit', 'motion sensor']),
    ('Drones & Action Cameras', ['gopro', 'drone', 'quadcopter']),
    ('Software', ['photoshop', 'illustrator', 'microsoft office']),
    ('Services', ['repair', 'installation', 'warranty', 'applecare']),
    ('Accessories & Cables', ['case', 'cable', 'adapter', 'charger', 'keyboard', 'mouse', 'lightning']),
]

def categorize_by_keyword(text):
    for category, keywords in category_rules:
        for kw in keywords:
            if re.search(rf'\b{re.escape(kw)}\b', text):
                return category
    return 'Other'

fehlende_maske = products_qu['category'].isna()
name_lower_fallback = products_qu.loc[fehlende_maske, 'desc'].fillna('').str.lower()
products_qu.loc[fehlende_maske, 'category'] = name_lower_fallback.apply(categorize_by_keyword)
python

# 5. Ergebnis prüfen
print(products_qu['category'].value_counts())
print('Anteil Other:', round((products_qu['category'] == 'Other').mean() * 100, 1), '%')

Optimierter code:

Was sich gegenüber eurer Version ändert:

type-Mapping zuerst (eindeutiger, s. letzte Analyse) — Keywords nur als Fallback für die restlichen ~20 %.
Wortgrenzen (\b...\b) statt reinem in-Substring → keine light/lightning, ram/frame, ios/studios-Kollisionen mehr.
1298/11905404 bewusst nicht geraten, landen im Keyword-Fallback bzw. ggf. 'Other'.

price/promo_price sind in products_qu.csv noch object (String) — falls ihr danach mit price rechnet (wie bei der > 500-Bedingung vorhin), erst pd.to_numeric() nicht vergessen.

In [ ]:
import re

# ============================================================
# 1. Primäre Zuordnung über 'type' (eindeutiger als reines Keyword-Matching)
#    Basis: manuelle Stichprobenprüfung der 30 größten type-Codes
# ============================================================
type_to_category = {
    '5,74E+15': 'Computers (iMac)',
    '1282':     'Computers (MacBook)',
    '12175397': 'Storage & NAS',
    '11865403': 'Accessories & Cables',
    '2158':     'Computers (MacBook)',
    '11935397': 'Storage & NAS',
    '1,02E+12': 'Computers (MacBook)',
    '12635403': 'Accessories & Cables',
    '13835403': 'Accessories & Cables',
    '1,44E+11': 'Services',
    '1364':     'Components (RAM/SSD)',
    '1433':     'Components (RAM/SSD)',
    '12585395': 'Accessories & Cables',
    '1296':     'Monitors',
    '1325':     'Accessories & Cables',
    '5384':     'Audio',
    '12215397': 'Components (RAM/SSD)',
    '5398':     'Audio',
    '57445397': 'Components (RAM/SSD)',
    '1334':     'Network',
    '1229':     'Accessories & Cables',
    '12655397': 'Storage & NAS',
    '2449':     'Wearables',
    '12995397': 'Accessories & Cables',
    '1515':     'Accessories & Cables',
    '13615399': 'Accessories & Cables',
    '1405':     'Tablets',
    '13555403': 'Accessories & Cables',
    # '1298' und '11905404' bewusst NICHT gemappt — Stichproben zeigten gemischte
    # Inhalte ohne eindeutiges Thema. Fallen unten in den Keyword-Fallback.
}

products_qu['category'] = products_qu['type'].map(type_to_category)

# ============================================================
# 2. Fallback über Keywords MIT Wortgrenzen (behebt light/lightning, ram/frame,
#    pen/opening, ios/studios etc. — reines "in text" hatte diese Kollisionen)
# ============================================================
category_rules = [
    ('Computers (MacBook)', ['macbook']),
    ('Computers (iMac)', ['imac']),
    ('Computers (Mac Mini)', ['mac mini']),
    ('Mobile Phones', ['iphone']),
    ('Tablets', ['ipad', 'wacom', 'intuos', 'graphics tablet', 'cintiq']),
    ('Wearables', ['airpods', 'watch', 'fitbit', 'bracelet']),
    ('Audio', ['speaker', 'headphone', 'headphones', 'microphone', 'soundbar']),
    ('Monitors', ['monitor', 'display', 'cinema display']),
    ('Storage & NAS', ['nas', 'synology', 'seagate', 'raid', 'qnap', 'hard drive']),
    ('Components (RAM/SSD)', ['ram', 'ssd', 'memory card', 'sata', 'flash drive', 'pendrive']),
    ('Network', ['router', 'ethernet', 'wifi', 'wi-fi', 'bluetooth', 'airport']),
    ('Smart Home', ['philips hue', 'thermostat', 'homekit', 'motion sensor']),
    ('Drones & Action Cameras', ['gopro', 'drone', 'quadcopter']),
    ('Software', ['photoshop', 'illustrator', 'microsoft office']),
    ('Services', ['repair', 'installation', 'warranty', 'applecare']),
    ('Accessories & Cables', ['case', 'cable', 'adapter', 'charger', 'keyboard',
                              'mouse', 'lightning', 'sleeve', 'stylus']),
]

def categorize_by_keyword(text):
    for category, keywords in category_rules:
        for kw in keywords:
            if re.search(rf'\b{re.escape(kw)}\b', text):
                return category
    return 'Other'

fehlende_maske = products_qu['category'].isna()
name_lower_fallback = products_qu.loc[fehlende_maske, 'desc'].fillna('').str.lower()
products_qu.loc[fehlende_maske, 'category'] = name_lower_fallback.apply(categorize_by_keyword)

# ============================================================
# 3. Kontrolle
# ============================================================
print(products_qu['category'].value_counts())
print('Anteil Other:', round((products_qu['category'] == 'Other').mean() * 100, 1), '%')